# Notebook 01 — Tổng quan dữ liệu

**Mục tiêu**: EDA sơ qua toàn bộ 15 bảng dữ liệu
- Shape, dtypes, missing values, unique counts
- Xác định các vấn đề chất lượng dữ liệu cần xử lý

**Tương ứng báo cáo**: Phần 2 — Chương 2 (Tổng quan dữ liệu và mô hình dữ liệu)

In [16]:
import sys
from pathlib import Path
sys.path.append('..')

import importlib
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

import src.data_loader as data_loader
import src.preprocessing as preprocessing
importlib.reload(data_loader)
importlib.reload(preprocessing)
load_all_raw = data_loader.load_all_raw
summarize = preprocessing.summarize

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.2f}'.format)
REPORT_DIR = Path('../outputs/figures/eda')
REPORT_DIR.mkdir(parents=True, exist_ok=True)
%matplotlib inline

## 1. Load tất cả dữ liệu

In [5]:
tables = load_all_raw()

for name, df in tables.items():
    print(f'{name:20s} | rows: {df.shape[0]:>8,} | cols: {df.shape[1]:>3}')

d:\Code\DS_Code\revenue_forecast_ecommerce\notebooks\..\src\data_loader.py:11: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(path)


products             | rows:    2,412 | cols:   8
customers            | rows:  121,930 | cols:   7
promotions           | rows:       50 | cols:  10
geography            | rows:   39,948 | cols:   4
orders               | rows:  646,945 | cols:   8
order_items          | rows:  714,669 | cols:   7
payments             | rows:  646,945 | cols:   4
shipments            | rows:  566,067 | cols:   4
returns              | rows:   39,939 | cols:   7
reviews              | rows:  113,551 | cols:   7
sales                | rows:    3,833 | cols:   3
inventory            | rows:   60,247 | cols:  17
web_traffic          | rows:    3,652 | cols:   7


## 2. Chất lượng dữ liệu từng bảng

In [17]:
table_overview = []
detail_frames = []

for name, df in tables.items():
    summary = summarize(df, name, verbose=False)
    summary_display = summary.reset_index().rename(columns={'index': 'column'})
    summary_display.insert(0, 'table', name)
    detail_frames.append(summary_display)
    summary_display.to_csv(REPORT_DIR / f'{name}_summary.csv', index=False)
    table_overview.append({
        'table': name,
        'rows': len(df),
        'cols': df.shape[1],
        'missing_cells': int(df.isna().sum().sum()),
        'missing_cols': int((df.isna().sum() > 0).sum()),
        'duplicate_rows': int(df.duplicated().sum()),
    })
    display(summary_display)

overview_df = pd.DataFrame(table_overview).sort_values(
    ['missing_cells', 'missing_cols', 'duplicate_rows', 'rows'], ascending=[False, False, False, False]
)
display(overview_df)

,table,column,dtype,null_count,null_pct,unique,sample
0,products,product_id,int64,0,0.00,2412,536
1,products,product_name,object,0,0.00,2172,SaigonFlex UC-01
2,products,category,object,0,0.00,4,Streetwear
3,products,segment,object,0,0.00,8,Everyday
4,products,size,object,0,0.00,4,S
5,products,color,object,0,0.00,10,green
6,products,price,float64,0,0.00,1990,"11,059.65"
7,products,cogs,float64,0,0.00,2381,"9,704.84"


,table,column,dtype,null_count,null_pct,unique,sample
0,customers,customer_id,int64,0,0.00,121930,1
1,customers,zip,int64,0,0.00,31491,15201
2,customers,city,object,0,0.00,42,Hai Phong
3,customers,signup_date,object,0,0.00,3941,2021-12-30
4,customers,gender,object,0,0.00,3,Female
5,customers,age_group,object,0,0.00,5,35-44
6,customers,acquisition_channel,object,0,0.00,6,social_media


,table,column,dtype,null_count,null_pct,unique,sample
0,promotions,promo_id,object,0,0.00,50,PROMO-0001
1,promotions,promo_name,object,0,0.00,50,Spring Sale 2013
2,promotions,promo_type,object,0,0.00,2,percentage
3,promotions,discount_value,float64,0,0.00,6,12.00
4,promotions,start_date,object,0,0.00,50,2013-03-18
5,promotions,end_date,object,0,0.00,50,2013-04-17
6,promotions,applicable_category,object,40,80.00,2,NaN
7,promotions,promo_channel,object,0,0.00,5,email
8,promotions,stackable_flag,int64,0,0.00,2,1
9,promotions,min_order_value,int64,0,0.00,5,0


,table,column,dtype,null_count,null_pct,unique,sample
0,geography,zip,int64,0,0.00,39948,15201
1,geography,city,object,0,0.00,42,Hai Phong
2,geography,region,object,0,0.00,3,East
3,geography,district,object,0,0.00,39,District #13


,table,column,dtype,null_count,null_pct,unique,sample
0,orders,order_id,int64,0,0.00,646945,1
1,orders,order_date,object,0,0.00,3833,2012-07-04
2,orders,customer_id,int64,0,0.00,90246,58578
3,orders,zip,int64,0,0.00,29932,1109
4,orders,order_status,object,0,0.00,6,delivered
5,orders,payment_method,object,0,0.00,5,credit_card
6,orders,device_type,object,0,0.00,3,desktop
7,orders,order_source,object,0,0.00,6,paid_search


...

,table,column,dtype,null_count,null_pct,unique,sample
0,reviews,review_id,object,0,0.00,113551,REV-0000001
1,reviews,order_id,int64,0,0.00,111369,1
2,reviews,product_id,int64,0,0.00,1412,2400
3,reviews,customer_id,int64,0,0.00,48676,58578
4,reviews,review_date,object,0,0.00,3825,2012-07-24
5,reviews,rating,int64,0,0.00,5,5
6,reviews,review_title,object,0,0.00,18,Highly recommend


,table,column,dtype,null_count,null_pct,unique,sample
0,sales,Date,object,0,0.00,3833,2012-07-04
1,sales,Revenue,float64,0,0.00,3833,"5,123,547.94"
2,sales,COGS,float64,0,0.00,3833,"3,982,991.19"


,table,column,dtype,null_count,null_pct,unique,sample
0,inventory,snapshot_date,object,0,0.00,126,2022-10-31
1,inventory,product_id,int64,0,0.00,1624,1
2,inventory,stock_on_hand,int64,0,0.00,1895,3
3,inventory,units_received,int64,0,0.00,360,1
4,inventory,units_sold,int64,0,0.00,303,1
5,inventory,stockout_days,int64,0,0.00,29,2
6,inventory,days_of_supply,float64,0,0.00,9289,90.00
7,inventory,fill_rate,float64,0,0.00,29,0.93
8,inventory,stockout_flag,int64,0,0.00,2,1
9,inventory,overstock_flag,int64,0,0.00,2,0


,table,column,dtype,null_count,null_pct,unique,sample
0,web_traffic,date,object,0,0.00,3652,2013-01-01
1,web_traffic,sessions,int64,0,0.00,3447,9760
2,web_traffic,unique_visitors,int64,0,0.00,3382,7253
3,web_traffic,page_views,int64,0,0.00,3620,39093
4,web_traffic,bounce_rate,float64,0,0.00,261,0.01
5,web_traffic,avg_session_duration_sec,float64,0,0.00,1771,102.90
6,web_traffic,traffic_source,object,0,0.00,6,organic_search


,table,rows,cols,missing_cells,missing_cols,duplicate_rows
5,order_items,714669,7,1152816,2,0
2,promotions,50,10,40,1,0
4,orders,646945,8,0,0,0
6,payments,646945,4,0,0,0
7,shipments,566067,4,0,0,0
1,customers,121930,7,0,0,0
9,reviews,113551,7,0,0,0
11,inventory,60247,17,0,0,0
3,geography,39948,4,0,0,0
8,returns,39939,7,0,0,0


## 3. Tổng hợp missing values — heatmap

In [9]:
# TODO: Vẽ heatmap missing values cho từng bảng có null
for name, df in tables.items():
    null_pct = df.isnull().sum() / len(df)
    if null_pct.max() > 0:
        print(f'\n{name}:')
        print(null_pct[null_pct > 0].sort_values(ascending=False).to_string())


promotions:
applicable_category   0.80

order_items:
promo_id_2   1.00
promo_id     0.61


## 4. Mô hình dữ liệu — ERD

Sơ đồ ERD đầy đủ + mô tả quan hệ: xem [`reports/erd/ERD.md`](../reports/erd/ERD.md).
Bản DBML để xuất ảnh chất lượng cao (dbdiagram.io): [`reports/erd/schema.dbml`](../reports/erd/schema.dbml).

```mermaid
erDiagram
    GEOGRAPHY  ||--o{ CUSTOMERS   : zip
    GEOGRAPHY  ||--o{ ORDERS      : zip
    CUSTOMERS  ||--o{ ORDERS      : customer_id
    CUSTOMERS  ||--o{ REVIEWS     : customer_id
    ORDERS     ||--|{ ORDER_ITEMS : order_id
    ORDERS     ||--o| PAYMENTS    : order_id
    ORDERS     ||--o| SHIPMENTS   : order_id
    ORDERS     ||--o{ RETURNS     : order_id
    ORDERS     ||--o{ REVIEWS     : order_id
    PRODUCTS   ||--o{ ORDER_ITEMS : product_id
    PRODUCTS   ||--o{ RETURNS     : product_id
    PRODUCTS   ||--o{ REVIEWS     : product_id
    PRODUCTS   ||--o{ INVENTORY   : product_id
    PROMOTIONS ||--o{ ORDER_ITEMS : promo_id
    ORDERS     }o--|| SALES       : order_date~Date
    WEB_TRAFFIC ||--o| SALES      : date~Date
```

→ **Phát hiện quan trọng**: `sales.csv` (target) là bảng **tổng hợp doanh thu theo ngày**, không nối trực tiếp khóa ngoại — phải roll-up từ `orders`/`order_items` và ghép theo trục thời gian với `web_traffic`, `promotions`, `inventory`.

## 5. Kết luận & Next steps

**TODO**: Tóm tắt các vấn đề chất lượng dữ liệu phát hiện được:
- Bảng nào có nhiều null nhất? 
=> Bảng order_items và promotions chứa null. 
+ order_items có 2 cột có giá trị null promo_id (438353) và promo_id_2 (714463) - nghiệp vụ không cần xử lý
+ promotions có cột applicable_category bị null 40 dòn - không cần xử lý, đề null là áp dụng cho tất cả danh mục
- Bảng nào có duplicate? 
=> Ko có bảng nào chứa duplicate
- Kiểu dữ liệu nào cần convert? 
=> Lỗi dữ liệu order_date, start_date, end_date => object --> datetime

→ Chuyển sang **Notebook 02** để xử lý.